In [ ]:
import os
import numpy as np
import nibabel.freesurfer.io as fsio
import plotly.graph_objects as go

In [ ]:
def load_fs_surface_defualt(subjects_dir, surface_name, surface_scale_name):
    surf_path= os.path.join(subjects_dir, surface_name)
    scalar_path = os.path.join(subjects_dir, surface_scale_name)
    verts, faces = fsio.read_geometry(surf_path)          # verts: (N,3), faces: (M,3)
    scalars = fsio.read_morph_data(scalar_path).astype(np.float32)  # (N,)
    return verts, faces, scalars

def plot_both_hemis_plotly(
    lh_verts, lh_faces, lh_scalars,
    rh_verts, rh_faces, rh_scalars,
    title="FS Surface (LH+RH)",
    value_range=None,                 # (vmin, vmax) 手动指定；None=自动
    percentile_range=(2, 98),
    symmetric=False,                  # True: [-m, m]
    background="black",               # "black" or "white"
    y_sign=+1,                        # 前后
    colorscale="Jet",
    explode=False,                    
    explode_gap=0,                 
):
    # ---- 统一值域：两半球一起算 ----
    s_all = np.concatenate([np.asarray(lh_scalars).ravel(), np.asarray(rh_scalars).ravel()])
    s_valid = s_all[np.isfinite(s_all)]
    if s_valid.size == 0:
        vmin, vmax = 0.0, 1.0
    elif value_range is not None:
        vmin, vmax = float(value_range[0]), float(value_range[1])
    else:
        if symmetric:
            m = float(np.percentile(np.abs(s_valid), percentile_range[1]))
            vmin, vmax = -m, m
        else:
            # vmin, vmax = np.percentile(s_valid, list(percentile_range)).astype(float)
            vmin, vmax = -4, 4

    # ---- 可选：exploded view----
    lhV = np.array(lh_verts, dtype=np.float32, copy=True)
    rhV = np.array(rh_verts, dtype=np.float32, copy=True)
    if explode:
        lhV[:, 0] -= explode_gap / 2.0
        rhV[:, 0] += explode_gap / 2.0

    # ---- 左前方光源：按整体尺寸自适应 ----
    V_all = np.vstack([lhV, rhV])
    center = V_all.mean(axis=0)
    scale = float(np.linalg.norm(V_all.max(axis=0) - V_all.min(axis=0)) + 1e-6)

    light_pos = dict(
        x=float(center[0] - 1.2 * scale),           # 左
        y=float(center[1] + y_sign * 1.2 * scale),  # 前（不对就改 y_sign=-1）
        z=float(center[2] + 0.8 * scale),           # 上
    )

    lighting = dict(ambient=0.12, diffuse=0.85, specular=0.25, roughness=0.45, fresnel=0.08)

    # ---- 画两个 trace：只保留一个 colorbar ----
    fig = go.Figure()

    fig.add_trace(go.Mesh3d(
        x=lhV[:, 0], y=lhV[:, 1], z=lhV[:, 2],
        i=lh_faces[:, 0], j=lh_faces[:, 1], k=lh_faces[:, 2],
        intensity=np.asarray(lh_scalars, dtype=np.float32),
        intensitymode="vertex",
        customdata = np.c_[lh_scalars],
        hovertemplate="curv:%{customdata[0]:.4f}<extra></extra>",
        colorscale=colorscale,
        cmin=vmin, cmax=vmax,
        showscale=False,  # LH 不显示色条
        lightposition=light_pos,
        lighting=lighting,
        name="RH",
    ))

    fig.add_trace(go.Mesh3d(
        x=rhV[:, 0], y=rhV[:, 1], z=rhV[:, 2],
        i=rh_faces[:, 0], j=rh_faces[:, 1], k=rh_faces[:, 2],
        intensity=np.asarray(rh_scalars, dtype=np.float32),
        customdata=np.c_[rh_scalars],         # ✅ 新增
        hovertemplate="curv:%{customdata[0]:.4f}<extra></extra>", 
        intensitymode="vertex",
        colorscale=colorscale,
        cmin=vmin, cmax=vmax,
        showscale=True,   # RH 显示色条（唯一一个）
        colorbar=dict(thickness=16, len=0.6, outlinewidth=0, ticks="outside"),
        lightposition=light_pos,
        lighting=lighting,
        name="LH",
    ))

    # ---- 背景/坐标轴 ----
    bg = "black" if str(background).lower() in ("black", "#000", "#000000") else "white"
    fg = "white" if bg == "black" else "black"

    fig.update_layout(
        title=dict(text=title, font=dict(color=fg)),
        scene=dict(
            aspectmode="data",
            bgcolor=bg,
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
        ),
        paper_bgcolor=bg,
        margin=dict(l=0, r=80, t=40, b=0),
    )

    return fig



In [ ]:
# ===== 用法 =====
subjects_dir = "/mnt/sda/public/00_Projects/2512_AutoRecon/data/surface/"


# ==== white matter =======
lh_path = "lh.smoothwm.nofix"
lh_scalars_path = "lh.smoothwm.nofix.H"
lh_verts, lh_faces, lh_scalars = load_fs_surface_defualt(subjects_dir, lh_path, lh_scalars_path)


surf_name_rh = "rh.smoothwm.nofix"
surf_scale_name_rh = "rh.smoothwm.nofix.H"
rh_verts, rh_faces, rh_scalars = load_fs_surface_defualt(subjects_dir, surf_name_rh, surf_scale_name_rh)

In [ ]:
fig = plot_both_hemis_plotly(
    lh_verts, lh_faces, lh_scalars,
    rh_verts, rh_faces, rh_scalars,
    percentile_range=(5, 95),
    symmetric=False,          
    background="white",
    explode=True,            
    explode_gap=0,
)
fig.show()

In [ ]:
# ===== Cotrex =====
lh_path = "lh.smoothcortex.nofix"
lh_scalars_path = "lh.smoothcortex.nofix.H"
lh_verts, lh_faces, lh_scalars = load_fs_surface_defualt(subjects_dir, lh_path, lh_scalars_path)


surf_name_rh = "rh.smoothcortex.nofix"
surf_scale_name_rh = "rh.smoothcortex.nofix.H"
rh_verts, rh_faces, rh_scalars = load_fs_surface_defualt(subjects_dir, surf_name_rh, surf_scale_name_rh)

fig = plot_both_hemis_plotly(
    lh_verts, lh_faces, lh_scalars,
    rh_verts, rh_faces, rh_scalars,
    percentile_range=(5, 95),
    symmetric=False,          
    background="white",
    explode=True,            
    explode_gap=0,
)
fig.show()